# Cats vs Dogs CNN Mini-Project
Binary image classification with real-world colour photos of different sizes.

Steps: check the data, clean bad files, build datasets, train a baseline CNN,
train a CNN with data augmentation, compare, and analyse errors.

In [ ]:
from pathlib import Path
import re
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras import Input, Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import (
    Rescaling, Conv2D, MaxPooling2D, Flatten, Dense, Dropout,
    RandomFlip, RandomRotation, RandomZoom,
)

# ---- Settings you may need to change --------------------------------------
# DATA_DIR must contain ONE SUB-FOLDER PER CLASS, for example:
#   data/PetImages/Cat/*.jpg
#   data/PetImages/Dog/*.jpg
DATA_DIR = "data/PetImages"
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 30              # upper limit; EarlyStopping usually stops sooner
MOVE_BAD_FILES = True    # move unreadable images to a "bad_images" folder (nothing is deleted)
# ----------------------------------------------------------------------------

SEED = 42
tf.keras.utils.set_random_seed(SEED)

## 1. Check the data folder
The notebook accepts either layout:
  * one sub-folder per class (Cat/, Dog/), at any depth under DATA_DIR
  * all images in one folder, with the class in the file name (cat.0.jpg, dog.0.jpg)

In [ ]:
root = Path(DATA_DIR)
assert root.exists(), f"Folder not found: {root.resolve()}  ->  update DATA_DIR"

print("Folder:", root.resolve())
print("Direct contents (first 10):", [p.name for p in list(root.iterdir())[:10]])

## 2. Find (and move) corrupt images
Real photo collections often contain a few broken files that would crash training.

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}


def find_bad_images(folder):
    bad = []
    for f in Path(folder).rglob("*"):
        if not f.is_file() or f.suffix.lower() not in IMG_EXT:
            continue
        try:
            if f.stat().st_size == 0:
                raise ValueError("empty file")
            with Image.open(f) as im:
                im.verify()
            with Image.open(f) as im:   # verify() is not enough for truncated files
                im.load()
        except Exception:
            bad.append(f)
    return bad


bad_files = find_bad_images(root)
print(f"Found {len(bad_files)} unreadable images")

if bad_files and MOVE_BAD_FILES:
    bad_dir = root.parent / "bad_images"
    bad_dir.mkdir(exist_ok=True)
    for f in bad_files:
        shutil.move(str(f), str(bad_dir / f"{f.parent.name}_{f.name}"))
    print(f"Moved them to {bad_dir}")

## 3. Build the datasets
80% train, 10% validation, 10% test. Images are resized to a fixed size;
scaling to 0-1 happens inside the model.

In [ ]:
def collect_images(folder):
    """Find every image and infer its class from the folder name or the file-name prefix."""
    files = sorted(f for f in Path(folder).rglob("*")
                   if f.is_file() and f.suffix.lower() in IMG_EXT and "bad_images" not in f.parts)
    if len({f.parent.name for f in files}) >= 2:
        names, how = [f.parent.name for f in files], "folder name"
    else:
        names, how = [re.split(r"[._\-\d]", f.name)[0].lower() for f in files], "file-name prefix"
    return files, names, how


files, names, how = collect_images(root)
counts = pd.Series(names).value_counts()
print(f"Found {len(files)} images; class taken from the {how}:")
print(counts.to_string())

assert len(counts) == 2, (
    f"Expected exactly 2 classes but found {len(counts)}: {list(counts.index)}. "
    "Point DATA_DIR at the folder that holds only the cat and dog images "
    "(for example not the unlabeled 'test' folder)."
)

CLASS_NAMES = sorted(counts.index)                       # label 0 / label 1
labels_all = np.array([CLASS_NAMES.index(n) for n in names])
print("Classes:", CLASS_NAMES, "-> label 0 =", CLASS_NAMES[0], ", label 1 =", CLASS_NAMES[1])

# 80% train / 10% validation / 10% test (stratified, fixed file lists)
idx = np.arange(len(files))
train_i, hold_i = train_test_split(idx, test_size=0.2, random_state=SEED, stratify=labels_all)
val_i, test_i = train_test_split(hold_i, test_size=0.5, random_state=SEED, stratify=labels_all[hold_i])
print("train:", len(train_i), "| validation:", len(val_i), "| test:", len(test_i))


def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    return img, label


def make_ds(indices, training=False):
    paths = [str(files[i]) for i in indices]
    labels = labels_all[indices].astype("float32")[:, None]   # shape (N, 1)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(paths), seed=SEED)                # reshuffled every epoch
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).ignore_errors()
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


def dataset_to_arrays(ds):
    """Read a dataset once into fixed arrays (used for the test set)."""
    xs, ys = [], []
    for x, y in ds:
        xs.append(x.numpy().astype("uint8"))
        ys.append(y.numpy().flatten().astype("int32"))
    return np.concatenate(xs), np.concatenate(ys)


train_ds = make_ds(train_i, training=True)
val_ds = make_ds(val_i)
X_test, y_test = dataset_to_arrays(make_ds(test_i))
print("test arrays:", X_test.shape, y_test.shape)

In [ ]:
def show_images(images, titles, rows, cols, figsize=(10, 6)):
    plt.figure(figsize=figsize)
    for i, (img, title) in enumerate(zip(images, titles)):
        if i >= rows * cols:
            break
        plt.subplot(rows, cols, i + 1)
        plt.imshow(np.asarray(img).astype("uint8"))
        plt.title(title, fontsize=9)
        plt.axis("off")
    plt.tight_layout()
    plt.show()


for images, labels in train_ds.take(1):
    show_images(images[:12], [CLASS_NAMES[int(l[0])] for l in labels[:12]], rows=3, cols=4)

## 4. Build the CNN
One function, two versions: `augment=False` (baseline) and `augment=True`.
Binary output: **1 neuron + sigmoid** gives P(class 1), trained with binary cross-entropy.

In [ ]:
def build_model(augment=False, dropout=0.5):
    model = Sequential(name="cnn_aug" if augment else "cnn_base")
    model.add(Input(shape=IMG_SIZE + (3,)))
    if augment:  # only active during training
        model.add(RandomFlip("horizontal"))
        model.add(RandomRotation(0.1))
        model.add(RandomZoom(0.1))
    model.add(Rescaling(1.0 / 255))
    for filters in (32, 64, 128, 128):
        model.add(Conv2D(filters, (3, 3), activation="relu"))
        model.add(MaxPooling2D((2, 2)))
    model.add(Flatten())
    model.add(Dense(256, activation="relu"))
    model.add(Dropout(dropout))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


model = build_model()
model.summary()

### Board exercise: feature-map sizes (128 x 128 input)

| Layer | Output shape |
|---|---|
| Conv(32) + Pool | 126x126x32 -> 63x63x32 |
| Conv(64) + Pool | 61x61x64 -> 30x30x64 |
| Conv(128) + Pool | 28x28x128 -> 14x14x128 |
| Conv(128) + Pool | 12x12x128 -> 6x6x128 |
| Flatten | 4608 |

Compare with the summary above.

## 5. Train the baseline (no augmentation)

In [ ]:
def plot_history(history, title=""):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, metric in zip(axes, ["accuracy", "loss"]):
        ax.plot(history.history[metric], label="train")
        ax.plot(history.history["val_" + metric], label="validation")
        ax.set_xlabel("epoch")
        ax.set_title(f"{title} {metric}".strip())
        ax.legend()
    plt.tight_layout()
    plt.show()


def new_early_stop():
    return EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)


history_base = model.fit(
    train_ds, epochs=EPOCHS, validation_data=val_ds,
    callbacks=[new_early_stop()], verbose=2,
)
plot_history(history_base, "baseline")

Typical pattern: training accuracy keeps rising while validation accuracy stalls.
That gap is **overfitting**, and it is the reason for the next step.

## 6. Train with data augmentation

In [ ]:
tf.keras.utils.set_random_seed(SEED)
model_aug = build_model(augment=True)

history_aug = model_aug.fit(
    train_ds, epochs=EPOCHS, validation_data=val_ds,
    callbacks=[new_early_stop()], verbose=2,
)
plot_history(history_aug, "augmented")

In [ ]:
# What augmentation does to one image
aug_layers = Sequential([RandomFlip("horizontal"), RandomRotation(0.1), RandomZoom(0.1)])
sample = X_test[:1]
show_images(
    [sample[0]] + [aug_layers(sample, training=True)[0].numpy() for _ in range(7)],
    ["original"] + [f"aug {i}" for i in range(1, 8)],
    rows=2, cols=4, figsize=(10, 5),
)

## 7. Evaluate both models on the test set

In [ ]:
def evaluate(model, name):
    probs = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0).flatten()
    y_pred = (probs > 0.5).astype("int32")
    acc = float((y_pred == y_test).mean())
    print(f"{name}: test accuracy = {acc:.4f}")
    return probs, y_pred, acc


probs_base, pred_base, acc_base = evaluate(model, "Baseline CNN")
probs_aug, pred_aug, acc_aug = evaluate(model_aug, "CNN + augmentation")

print()
print(classification_report(y_test, pred_aug, target_names=CLASS_NAMES, digits=3))

In [ ]:
def plot_confusion(y_true, y_hat, title):
    cm = confusion_matrix(y_true, y_hat)
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    axes[0].set_title("Counts")
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", ax=axes[1],
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    axes[1].set_title("Normalized by actual class")
    for ax in axes:
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_confusion(y_test, pred_base, "Baseline CNN")
plot_confusion(y_test, pred_aug, "CNN + augmentation")

## 8. Error analysis (augmented model)

In [ ]:
wrong = np.where(y_test != pred_aug)[0]
print(f"{len(wrong)} wrong predictions out of {len(y_test)}")

show_images(
    X_test[wrong[:12]],
    [f"True: {CLASS_NAMES[y_test[i]]}\nPred: {CLASS_NAMES[pred_aug[i]]} (p={probs_aug[i]:.2f})"
     for i in wrong[:12]],
    rows=3, cols=4, figsize=(12, 9),
)

In [ ]:
# Confidently wrong: the model was sure, and still wrong
confidence = np.where(pred_aug == 1, probs_aug, 1 - probs_aug)
confident_wrong = wrong[np.argsort(-confidence[wrong])][:12]

show_images(
    X_test[confident_wrong],
    [f"True: {CLASS_NAMES[y_test[i]]}\nPred: {CLASS_NAMES[pred_aug[i]]} ({confidence[i]:.0%})"
     for i in confident_wrong],
    rows=3, cols=4, figsize=(12, 9),
)

## 9. Results table

In [ ]:
results = pd.DataFrame([
    {"model": "CNN", "params": model.count_params(),
     "epochs run": len(history_base.history["loss"]),
     "best val acc": max(history_base.history["val_accuracy"]), "test acc": acc_base},
    {"model": "CNN + augmentation", "params": model_aug.count_params(),
     "epochs run": len(history_aug.history["loss"]),
     "best val acc": max(history_aug.history["val_accuracy"]), "test acc": acc_aug},
])
results

## 10. Predict a new image

In [ ]:
def predict_image(path, model=model_aug):
    img = tf.keras.utils.load_img(path, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)[np.newaxis]
    p = float(model.predict(arr, verbose=0)[0, 0])
    label = CLASS_NAMES[int(p > 0.5)]
    conf = p if p > 0.5 else 1 - p
    plt.imshow(img)
    plt.title(f"{label} ({conf:.0%})")
    plt.axis("off")
    plt.show()


# predict_image("my_pet.jpg")     # <- put the path of your own photo here

In [ ]:
model_aug.save("cats_vs_dogs_cnn.keras")
print("saved to cats_vs_dogs_cnn.keras")

## Discussion questions
1. How large is the gap between training and validation accuracy for the baseline? What does it mean?
2. Did augmentation shrink that gap? Did test accuracy improve?
3. Why do we use 1 sigmoid neuron here but 10 softmax neurons for MNIST and CIFAR-10?
4. Look at the confidently wrong photos. What might have confused the model (background, pose, several animals)?
5. What would you change next: a bigger image size, a deeper CNN, or transfer learning?